# RAG Frameworks

**Module:** 04 — RAG

Compare LangChain, LlamaIndex, Haystack, and custom pipelines—when to adopt or DIY.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe core abstractions in major RAG frameworks
- Choose framework vs custom for a given team constraint
- Sketch equivalent pipelines across tools
- Avoid framework lock-in anti-patterns


## LangChain

**Definition.** Ecosystem of chains/runnables, retrievers, and integrations for LLM apps.

**Why it matters.** Huge integration surface and rapid prototyping for RAG/agents.

**How it works.** Compose retrievers, prompt templates, and LCEL/runnables into chains.

**Intuition.** Batteries-included plumbing with many adapters.

**Common pitfalls.**
- Abstraction churn
- Hidden complexity in production debugging

**When to use.** Fast prototypes and broad tool integrations.

```mermaid
flowchart LR
  Q --> Retriever --> Prompt --> Model --> Out
```


In [ ]:
# Demo 1 — pseudo-LCEL style composition
def runnable(pipe):
    def inv(x):
        for step in pipe: x = step(x)
        return x
    return inv
chain = runnable([
    lambda q: {"query": q, "docs": [f"doc about {q}"]},
    lambda s: {**s, "prompt": f"ctx={s['docs']} q={s['query']}"},
    lambda s: {**s, "answer": "grounded answer"},
])
print(chain("refunds"))


In [ ]:
# Demo 2 — retriever interface duck-type
class Retriever:
    def invoke(self, query: str):
        return [{"page_content": f"hit for {query}", "metadata": {"id": "C1"}}]
print(Retriever().invoke("shipping"))


In [ ]:
# Demo 3 — env key placeholder
import os
YOUR_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY")
print("langchain would use key:", YOUR_API_KEY[:8] + "...")


### Try it yourself — LangChain

1. Map Basic RAG stages to LangChain classes you would use.


## LlamaIndex

**Definition.** Data framework centered on indexes, nodes, query engines, and RAG workflows.

**Why it matters.** Strong document-index abstractions and query engines for RAG.

**How it works.** Parse docs → nodes → index → query_engine.query(...).

**Intuition.** Index-first mental model.

**Common pitfalls.**
- Heavy magic defaults
- Need to understand Node metadata

**When to use.** Doc-heavy RAG with varied indexes (vector, keyword, KG).


In [ ]:
# Demo 1 — node/index/query mental model
nodes = [{"id": "n1", "text": "Refunds 60 days", "meta": {"parent": "p1"}}]
index = {"nodes": nodes}
def query_engine(q, index, k=1):
    return [n for n in index["nodes"] if any(w in n["text"].lower() for w in q.lower().split())][:k]
print(query_engine("refunds", index))


In [ ]:
# Demo 2 — response object shape
print({"response": "60 days", "source_nodes": [{"id": "n1", "score": 0.88}]})


### Try it yourself — LlamaIndex

1. Contrast Node vs your Chunk dataclass fields.


## Haystack

**Definition.** Pipeline-oriented NLP/RAG framework with explicit components and evaluation tooling.

**Why it matters.** Clear pipelines and production-oriented components.

**How it works.** Wire components (converters, retrievers, generators) into Pipelines.

**Intuition.** Directed graph of named components.

**Common pitfalls.**
- Verbosity for tiny demos
- Version migration

**When to use.** Teams wanting explicit pipelines and eval hooks.


In [ ]:
# Demo 1 — named component pipeline
components = {
  "retriever": lambda q: [f"doc:{q}"],
  "prompt": lambda docs, q: f"docs={docs}|q={q}",
  "generator": lambda prompt: f"ANSWER({prompt[:40]}...)",
}
def run(q):
    docs = components["retriever"](q)
    prompt = components["prompt"](docs, q)
    return components["generator"](prompt)
print(run("refund"))


In [ ]:
# Demo 2 — pipeline YAML-ish config
config = {"nodes": ["retriever", "reranker", "prompt", "generator"], "edges": "sequential"}
print(config)


### Try it yourself — Haystack

1. Draw a Haystack-style graph for hybrid + rerank + generate.


## Custom Pipelines

**Definition.** Hand-rolled orchestration with your services, observability, and contracts.

**Why it matters.** Maximum control, stable APIs, and clearer performance accounting.

**How it works.** Small pure functions/services + queue + metrics; frameworks optional at edges.

**Intuition.** Own the critical path; borrow adapters sparingly.

**Common pitfalls.**
- Rebuilding everything
- No eval harness

**When to use.** Strict SLOs, unusual data paths, or platform teams.

### Framework selection

| Need | Lean to |
|------|---------|
| Speed to demo | LangChain / LlamaIndex |
| Explicit pipelines | Haystack / custom |
| Deep index types | LlamaIndex |
| Strict control/SLO | Custom |


In [ ]:
# Demo 1 — minimal custom pipeline class
from dataclasses import dataclass

@dataclass
class CustomRAG:
    search: callable
    generate: callable
    def ask(self, q: str) -> dict:
        hits = self.search(q)
        ans = self.generate(q, hits)
        return {"answer": ans, "hits": hits}

rag = CustomRAG(search=lambda q: ["C1"], generate=lambda q,h: f"{q} -> {h}")
print(rag.ask("refunds"))


In [ ]:
# Demo 2 — boundary: framework only for SDKs
def embed_with_vendor(texts, api_key="YOUR_API_KEY"):
    return {"model": "emb", "n": len(texts), "api_key_prefix": api_key[:8]}
print(embed_with_vendor(["a", "b"]))


In [ ]:
# Demo 3 — observability hooks
def with_trace(name, fn):
    def wrapped(*a, **k):
        print("start", name); out = fn(*a, **k); print("end", name); return out
    return wrapped
@with_trace("retrieve")
def retrieve(q): return [q]
print(retrieve("x"))


### Try it yourself — Custom Pipelines

1. List 5 interfaces you would stabilize in a custom RAG platform.
2. Decide framework vs custom for a regulated bank assistant—justify.


## Glossary

- **LCEL**: LangChain Expression Language composition
- **query engine**: LlamaIndex high-level ask interface


### Workshop drill — RAG Frameworks (1)

Diagram the data flow on paper, then implement one missing log line per stage.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 1 — RAG Frameworks
stages = ['ingest','chunk','embed','retrieve','pack','generate']
for s in stages:
    print(f'log.{{s}}.ok = ?')


### Workshop drill — RAG Frameworks (2)

Create two adversarial queries (one ID-heavy, one paraphrase-heavy) and compare hits.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 2 — RAG Frameworks
queries = ['error code E42-refund', 'how do I get my money back?']
for q in queries:
    print('Q:', q)
    print('  TODO: print top-3 ids')


### Workshop drill — RAG Frameworks (3)

Write a refusal test: empty hits must not produce a confident numeric answer.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 3 — RAG Frameworks
def must_refuse(hits):
    return (not hits) or hits[0].get('score',0) < 0.2
assert must_refuse([])
assert must_refuse([{'score': 0.05}])
assert not must_refuse([{'score': 0.9}])
print('refusal tests ok')


### Workshop drill — RAG Frameworks (4)

Estimate cost: vary top_k and context tokens; print a small table.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 4 — RAG Frameworks
rows = []
for k in [2,4,8,16]:
    toks = k*400
    rows.append((k, toks, round(toks/1e6*0.5, 5)))
print('k  ctx_toks  approx_$')
for r in rows:
    print(*r)


## Summary & Key Takeaways

- Frameworks accelerate; they do not replace retrieval science
- Prefer clear stage contracts over magic chains
- Custom pipelines win on control; borrow vendor SDKs at edges
- Evaluate with the same metrics regardless of framework

### Practice

Reimplement Basic RAG once in 'framework pseudo' and once in CustomRAG.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
